# 51 · Feature & ML — MLflow tracking + model registry

**MLflow is the mesh's memory of every model it has trained.** A training run produces
numbers you need to keep — the hyperparameters that were tried, the accuracy that came
out, the fitted model itself — and MLflow is where all three land, so a result is never
just a line in a log that scrolls away. It answers two different questions with two
different surfaces:

> **Tracking** remembers *how each run went* — params in, metrics out, artifacts kept.
> **The registry** remembers *which model is the one to use* — named, versioned, loadable.

Those are not the same thing, and the split is the whole point of this notebook:

- **Tracking** is the lab notebook. Every run of every experiment writes its parameters,
  its metrics, and its artifacts under an experiment name. You browse it to compare runs
  and see what a sweep discovered. It is *append-only history*.
- **The registry** is the shelf. When a run produces a model worth keeping, it is
  *registered* under a stable name and gets a version number. You load a model *by name
  and version* — `models:/genre_classifier/9` — without knowing which run made it. It is
  *the curated current state*.

### Where MLflow sits — and where the training actually happens

This lab does not train inside the notebook, and this notebook does not pretend to. The
heavy compute runs **on rogueone** (the training box — 32 cores, an RTX-class GPU), in an
ephemeral **genre-trainer** container submitted to **Ray**. That container reads features
from the lakehouse, fits models, and **logs its runs and registers its models to *this*
MLflow** over the LAN. By the time you open this notebook, the results are already here.

> **Train on rogueone (Ray) → log to MLflow → read it here.** The notebook is the *read
> surface* on that flow. Section 6 explains the Ray training pattern in full; every run and
> model you browse below is its output. We deliberately do **not** run a live Ray client
> here — that client is version-pinned and token-authed, and reaching for it would make the
> demo about Ray plumbing instead of about MLflow. The honest, robust thing is to read what
> the training already logged.

> **Read-only.** Everything here is `search_experiments` / `search_runs` /
> `search_registered_models` / `load_model`. We create no runs, no experiments, no model
> versions — nothing is written to MLflow. Because we create nothing, there is **no cleanup
> section** (the same stance as the query notebooks `20`/`22`).

## Setup

The `mlflow` client is **not** baked into the singleuser base image (which ships `polars`,
`s3fs`, `pyarrow`, `duckdb`, `fastavro`), so we install it here. `polars` — used to render
every result frame, exactly as in the query notebooks `20`/`22` — already ships in the
image.

In [1]:
%pip install -q mlflow

Note: you may need to restart the kernel to use updated packages.


## Connect

Connection is **env-driven**, the same pattern every notebook in this library uses. The
committed default is the **in-cluster** tracking service URL
(`mlflow.weyland.svc.cluster.local:5000`); a validation run overrides `MLFLOW_TRACKING_URI`
via env (e.g. to the LAN NodePort) **without editing the notebook**.

This MLflow is behind **basic auth**. The client reads `MLFLOW_TRACKING_USERNAME` /
`MLFLOW_TRACKING_PASSWORD` straight from the environment — the singleuser pod injects them
from a sealed secret, so no credential is ever written into a cell. We default the username
to `admin` and take the password only from the environment; **we never echo the resolved
URI or the password**, so this notebook's saved output can't leak either. A connection is
proven by the **experiment count** it reads back, not by printing the address it read.

In [2]:
import os
import mlflow
from mlflow.tracking import MlflowClient

# username defaults to admin; password comes ONLY from the environment (never a literal here)
os.environ.setdefault("MLFLOW_TRACKING_USERNAME", os.environ.get("MLFLOW_TRACKING_USERNAME", "admin"))

# committed default = in-cluster tracking service; a validation run overrides MLFLOW_TRACKING_URI via env
mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://mlflow.weyland.svc.cluster.local:5000"))
client = MlflowClient()

# proves the connection by what it reads back, NOT by echoing the endpoint or the password
experiments = client.search_experiments()
print("connected — MLflow client", mlflow.__version__)
print("experiments visible:", len(experiments))

connected — MLflow client 3.15.2
experiments visible: 28


## Experiments — the lab notebook's table of contents

An **experiment** is a named bucket of runs. `search_experiments()` lists them; counting the
runs in each shows where the training effort actually went. We render it as a polars frame
(house style), sorted by run count so the busy experiments rise to the top.

Read this table as a map of the platform's ML activity: the LLM-gateway probe experiments
(`gateway/*`, `gateway-eval`), the RAG-eval history (`weyland_rag_eval`), and — the one this
notebook builds on — **`genre-classifier`**, the music-genre model whose Ray-tuned runs
dominate the count.

In [3]:
import polars as pl

# count runs per experiment (bounded fetch — plenty for a display table)
def run_count(experiment_id):
    return len(client.search_runs([experiment_id], max_results=5000))

rows = [(e.name, e.experiment_id, run_count(e.experiment_id)) for e in experiments]
exp_df = (
    pl.DataFrame(rows, schema=["experiment", "id", "n_runs"], orient="row")
      .sort("n_runs", descending=True)
)
print(f"{exp_df.height} experiments; {exp_df['n_runs'].sum()} runs total")
exp_df.head(15)

28 experiments; 400 runs total


experiment,id,n_runs
str,str,i64
"""genre-classifier""","""5""",247
"""weyland_rag_eval""","""2""",101
"""gateway-eval""","""29""",30
"""mlflow_evaluate""","""10""",12
"""gateway/openai-gpt-5-mini""","""18""",4
…,…,…
"""gateway/xai-grok-3-mini""","""26""",0
"""gateway/openrouter-meta-llama-…","""25""",0
"""gateway/together-meta-llama-Ll…","""24""",0


## Runs — params in, metrics out

Inside an experiment, each **run** records the parameters it used and the metrics it
produced. That is tracking's core value: you can line runs up and see *what changed and what
it bought you*. We open **`genre-classifier`** because it is the training experiment whose
results tie straight back to the Ray pattern (section 6) — a RandomForest that predicts a
track's genre from its audio features.

`search_runs` hands back each run's params and metrics; we lift the fields we care about into
a polars frame. First, the shape of the experiment — grouping by the `mode` param sorts the
runs into the kinds the trainer produces.

In [4]:
# resolve the training experiment live (don't assume its id)
genre_exp = client.get_experiment_by_name("genre-classifier")
genre_runs = client.search_runs([genre_exp.experiment_id], max_results=5000)
print(f"genre-classifier: {len(genre_runs)} runs (experiment id {genre_exp.experiment_id})")

# the `mode` param separates the three kinds of run this experiment holds
mode_counts = {}
for r in genre_runs:
    m = r.data.params.get("mode", "(none)")
    mode_counts[m] = mode_counts.get(m, 0) + 1
pl.DataFrame(sorted(mode_counts.items()), schema=["mode", "n_runs"], orient="row").sort("n_runs", descending=True)

genre-classifier: 247 runs (experiment id 5)


mode,n_runs
str,i64
"""ray-tune""",237
"""ray-tune-best""",8
"""(none)""",1
"""single""",1


The three meaningful shapes of run are exactly the Ray-Tune pattern of section 6 (a lone
early run predates the `mode` param and shows up as `(none)`):

- **`ray-tune`** — one run per hyperparameter *trial*. A sweep of dozens of trials, each a
  different `(n_estimators, max_depth, max_features, min_samples_leaf)` combination, each
  logging its own params + metrics. This is the bulk of the count.
- **`ray-tune-best`** — the *winner* of a sweep, retrained on the full split and **registered**
  to the model registry. One of these per completed sweep.
- **`single`** — a plain one-off fit (no sweep).

Now pull the runs into a frame — the sweep trials with their sampled hyperparameters and the
`f1_macro` / `accuracy` each achieved — and rank them, so the frame answers *which config the
sweep found best*.

In [5]:
# lift params + metrics of the sweep trials into a polars frame, ranked by f1_macro
def trial_row(r):
    p, m = r.data.params, r.data.metrics
    return (
        r.info.run_name,
        p.get("mode"),
        int(p["n_estimators"]) if "n_estimators" in p else None,
        int(p["max_depth"]) if "max_depth" in p else None,
        p.get("max_features"),
        int(p["min_samples_leaf"]) if "min_samples_leaf" in p else None,
        round(m["f1_macro"], 4) if "f1_macro" in m else None,
        round(m["accuracy"], 4) if "accuracy" in m else None,
    )

cols = ["run_name", "mode", "n_estimators", "max_depth", "max_features",
        "min_samples_leaf", "f1_macro", "accuracy"]
trials = [trial_row(r) for r in genre_runs if r.data.params.get("mode") == "ray-tune"]
trials_df = (
    pl.DataFrame(trials, schema=cols, orient="row")
      .drop_nulls("f1_macro")
      .sort("f1_macro", descending=True)
)
print(f"{trials_df.height} ray-tune trials; best f1_macro at the top")
trials_df.head(10)

237 ray-tune trials; best f1_macro at the top


run_name,mode,n_estimators,max_depth,max_features,min_samples_leaf,f1_macro,accuracy
str,str,i64,i64,str,i64,f64,f64
"""tune-trial""","""ray-tune""",200,20,"""log2""",1,0.3137,0.3287
"""tune-trial""","""ray-tune""",200,20,"""log2""",1,0.3137,0.3287
"""tune-trial""","""ray-tune""",200,20,"""log2""",1,0.3115,0.3287
"""tune-trial""","""ray-tune""",200,20,"""log2""",1,0.3115,0.3287
"""tune-trial""","""ray-tune""",200,20,"""log2""",1,0.3115,0.3287
"""tune-trial""","""ray-tune""",200,20,"""sqrt""",1,0.3115,0.3287
"""tune-trial""","""ray-tune""",200,20,"""log2""",1,0.3115,0.3287
"""tune-trial""","""ray-tune""",200,20,"""sqrt""",1,0.3115,0.3287
"""tune-trial""","""ray-tune""",200,20,"""sqrt""",1,0.3115,0.3287


That frame is tracking earning its keep: **every trial the sweep ran is still here**, so
you can see not just the winning hyperparameters but the whole shape of the search — which
knobs moved `f1_macro` (deeper trees at `max_depth=20` cluster at the top) and which barely
mattered. Now the other end of the experiment: the **registered winners**, one per sweep, each
the `ray-tune-best` run that was promoted to the registry.

In [6]:
# the runs that were retrained-and-registered — the models on the shelf came from these
def best_row(r):
    p, m = r.data.params, r.data.metrics
    return (
        r.info.run_name,
        p.get("feature_source"),
        int(p["n_classes"]) if "n_classes" in p else None,
        int(p["n_rows"]) if "n_rows" in p else None,
        round(m["f1_macro"], 4) if "f1_macro" in m else None,
        round(m["accuracy"], 4) if "accuracy" in m else None,
    )

best_cols = ["run_name", "feature_source", "n_classes", "n_rows", "f1_macro", "accuracy"]
bests = [best_row(r) for r in genre_runs if r.data.params.get("mode") == "ray-tune-best"]
pl.DataFrame(bests, schema=best_cols, orient="row").sort("f1_macro", descending=True)

run_name,feature_source,n_classes,n_rows,f1_macro,accuracy
str,str,i64,i64,f64,f64
"""genre-feast-tuned""","""feast""",113,89741,0.3137,0.3287
"""genre-silver-tuned""","""silver""",113,89741,0.3115,0.3287
"""genre-silver-tuned""","""silver""",113,89741,0.3115,0.3287
"""genre-silver-tuned""","""silver""",113,89741,0.3115,0.3287
"""genre-mart-tuned""","""mart""",113,89741,0.3108,0.3273
"""genre-silver-tuned""","""silver""",113,89741,0.308,0.3266
"""genre-silver-tuned""","""silver""",113,89741,0.308,0.3266
"""genre-silver-tuned""","""silver""",113,89741,0.308,0.3266


Each row is a full-dataset RandomForest over **11 audio features** across **~113 genre
classes** — the `feature_source` column shows the three feed variants the trainer supports
(`silver` hand-cleaned, `feast` point-in-time, `mart` the tested dbt mart). These are the runs
that produced the versions on the registry shelf, which is where we go next.

## Registry — the models on the shelf

`search_registered_models()` lists the named models; `search_model_versions(...)` lists the
versions under each name. Where tracking is append-only history, the registry is the *curated
current state* — you load a model by name and version and never think about which run made it.

In [7]:
reg_models = client.search_registered_models()
print("registered models:", [m.name for m in reg_models])

if reg_models:
    vrows = []
    for m in reg_models:
        for v in client.search_model_versions(f"name='{m.name}'"):
            vrows.append((m.name, int(v.version), v.current_stage, v.status, (v.run_id or "")[:12]))
    versions_df = (
        pl.DataFrame(vrows, schema=["name", "version", "stage", "status", "run_id"], orient="row")
          .sort(["name", "version"], descending=[False, True])
    )
    print(f"{versions_df.height} versions across {len(reg_models)} model(s)")
else:
    versions_df = pl.DataFrame(schema=["name", "version", "stage", "status", "run_id"])
    print("registry is empty — no models registered yet")
versions_df

registered models: ['genre_classifier']
9 versions across 1 model(s)


name,version,stage,status,run_id
str,i64,str,str,str
"""genre_classifier""",9,"""None""","""READY""","""4d51c522c701"""
"""genre_classifier""",8,"""None""","""READY""","""1bdc68943827"""
"""genre_classifier""",7,"""None""","""READY""","""d529c673804a"""
"""genre_classifier""",6,"""None""","""READY""","""edd5c5e9e453"""
"""genre_classifier""",5,"""None""","""READY""","""d698d8e6600b"""
"""genre_classifier""",4,"""None""","""READY""","""4017294b56b4"""
"""genre_classifier""",3,"""None""","""READY""","""f74decadc740"""
"""genre_classifier""",2,"""None""","""READY""","""7a2a9c8d0570"""
"""genre_classifier""",1,"""None""","""READY""","""304a820761de"""


Each **version** points back at the run that produced it (`run_id`) and carries a
`stage` (here `None` — this lab doesn't gate models through Staging/Production transitions).
The highest version number under a name is simply the most recently registered winner.

### Load a registered model and predict

The payoff of the registry: **load a model by name and version and use it**, with no
knowledge of how it was trained. Below we load the latest `genre_classifier` and predict a
genre for a **hand-built track** — the same 11 audio features the trainer fits on
(`danceability`, `energy`, `key`, `loudness`, `mode`, `speechiness`, `acousticness`,
`instrumentalness`, `liveness`, `valence`, `tempo`), in that order.

One honest caveat, guarded explicitly: the model *blob* lives in the artifact store
(MinIO's `s3://mlflow/...`), which is a **different credential** than the tracking API we
connected with. Browsing runs and the registry needs only the tracking auth we already have;
**downloading a model to load it** additionally needs the artifact-store S3 credentials
(`MLFLOW_S3_ENDPOINT_URL` + `AWS_*`). Where those aren't present in the environment, the load
can't complete — so the whole load-and-predict is wrapped in `try/except` and **degrades to a
note instead of an error**. The registry *metadata* above is the durable read; the live load
is a bonus that lights up wherever the artifact creds are wired.

In [8]:
import numpy as np

# the 11 audio features the genre-trainer fits on, IN ORDER (positional — the model was fit on a bare ndarray)
AUDIO = ["danceability", "energy", "key", "loudness", "mode", "speechiness",
         "acousticness", "instrumentalness", "liveness", "valence", "tempo"]

# a hand-built track: upbeat, danceable, major key — plausible audio-feature values
track = {
    "danceability": 0.82, "energy": 0.90, "key": 7, "loudness": -4.2, "mode": 1,
    "speechiness": 0.06, "acousticness": 0.02, "instrumentalness": 0.0,
    "liveness": 0.12, "valence": 0.85, "tempo": 124.0,
}
X = np.array([[track[f] for f in AUDIO]])   # shape (1, 11), features in trainer order

MODEL_NAME = "genre_classifier"
names = [m.name for m in reg_models]
if MODEL_NAME in names:
    latest = max(int(v.version) for v in client.search_model_versions(f"name='{MODEL_NAME}'"))
    uri = f"models:/{MODEL_NAME}/{latest}"
    try:
        model = mlflow.pyfunc.load_model(uri)          # downloads the artifact from the S3 store
        pred = model.predict(X)
        print(f"loaded {uri} and predicted:")
        print("  predicted genre:", pred[0])
    except Exception as e:
        # artifact-store creds absent (or unreachable) -> degrade to a note, never an error output
        print(f"model {uri} is registered, but the artifact could not be loaded here.")
        print("  loading the model blob needs the MinIO artifact-store creds")
        print("  (MLFLOW_S3_ENDPOINT_URL + AWS_*), which this environment doesn't carry.")
        print("  the registry metadata above is the durable read; wire those creds to load.")
        print(f"  ({type(e).__name__})")
else:
    # registry empty: show how a run's model artifact WOULD be registered (read-only — we don't run it)
    print(f"no '{MODEL_NAME}' registered. A run's logged model is promoted to the registry with:")
    print("  mlflow.register_model('runs:/<run_id>/model', 'genre_classifier')")
    print("  # or at log time: mlflow.sklearn.log_model(clf, 'model', registered_model_name='genre_classifier')")

model models:/genre_classifier/9 is registered, but the artifact could not be loaded here.
  loading the model blob needs the MinIO artifact-store creds
  (MLFLOW_S3_ENDPOINT_URL + AWS_*), which this environment doesn't carry.
  the registry metadata above is the durable read; wire those creds to load.
  (NoCredentialsError)


## Where the runs come from — the Ray training pattern

Everything above was *read*. This section explains what *wrote* it, because the runs and
versions only make sense once you know the machine that produced them — and that machine is
**not** this notebook or the k3s cluster. It is an ephemeral container on **rogueone**, the
lab's training box, driven by **Ray**.

The **genre-trainer** job runs like this:

1. **rogueone pulls the trainer image** from the in-cluster MinIO registry and starts it as a
   container. This is a separate box from the mesh — chosen for its cores and GPU — reaching
   the platform over the LAN.
2. **It reads features from the lakehouse.** The training set (`track_id` + the 11 audio
   features + `track_genre`) is materialized to lakeFS parquet by a meshed Dagster asset —
   from hand-cleaned `silver`, a `feast` point-in-time join, or the tested dbt `mart`. The
   trainer reads it straight from the lakeFS S3 gateway.
3. **Ray Tune runs the sweep.** A local Ray cluster on rogueone's cores fans a hyperparameter
   search across trials (`{n_estimators, max_depth, max_features, min_samples_leaf}`), ~8
   concurrent at 4 CPUs each. **Every trial logs its own MLflow run** — those are the dozens
   of `ray-tune` rows you ranked in section 4.
4. **The winner is retrained on a worker and registered.** The best-`f1_macro` config is
   retrained on the full split *on a Ray worker* (so the multi-GB forest never OOMs the head),
   logged as the `ray-tune-best` run, and **registered as `genre_classifier`** — the versions
   you listed in section 5.

The design detail worth carrying away is the **two MLflow planes**, because it's why the
metadata was trivially readable above while the model load needed extra creds:

> **Metadata** (params, metrics, the registry entry) → the MLflow **tracking server** →
> Postgres. Small, always fast, and all this notebook needed to browse everything.
> **Artifact** (the fitted model blob) → uploaded **direct to MinIO** (`s3://mlflow/...`),
> bypassing the tracking pod's artifact relay so a large `model.pkl` never has to squeeze
> through it. That's the plane the section-5 load reaches into — a different credential.

So the counts line up exactly with what you read: a sweep of many `ray-tune` trials, a
handful of `ray-tune-best` winners promoted to the registry, and one `genre_classifier` name
carrying a version per winner. The Ray client that drove it is version-pinned and token-authed
and lives with the trainer on rogueone — which is precisely why this notebook reads its
*results* rather than trying to reproduce its *plumbing*.

In [9]:
# the Ray pattern's fingerprint, read straight off the runs: sweep trials vs registered winners
n_trials = sum(1 for r in genre_runs if r.data.params.get("mode") == "ray-tune")
n_best = sum(1 for r in genre_runs if r.data.params.get("mode") == "ray-tune-best")
n_registered = len(client.search_model_versions("name='genre_classifier'"))
print(f"ray-tune trials logged (one MLflow run each) : {n_trials}")
print(f"ray-tune-best winners (retrained + registered): {n_best}")
print(f"genre_classifier versions on the registry     : {n_registered}")

ray-tune trials logged (one MLflow run each) : 237
ray-tune-best winners (retrained + registered): 8
genre_classifier versions on the registry     : 9


## When to reach for MLflow

MLflow is two tools wearing one name; reach for the half that fits the question.

| you want to… | use | in this notebook |
|--------------|-----|------------------|
| compare training runs — what params, what metrics, what a sweep found | **tracking** (`search_runs`) | sections 3–4 |
| keep a durable, versioned handle on the model to *use* | **registry** (`search_registered_models` / `load_model`) | section 5 |
| know *what produced* the runs and models | the **Ray genre-trainer** on rogueone | section 6 |

**Reach for tracking** whenever a training process makes choices you'll want to justify or
revisit — every run's params and metrics are kept, so a sweep is auditable months later and
two approaches can be compared on the same axes. It is the antidote to results that live only
in a log.

**Reach for the registry** when a model needs to be *used* by something that shouldn't know
how it was trained — a serving job, an evaluation, another notebook. `models:/name/version`
is a stable handle; the training run behind it can change without the consumer changing.

**And remember the division of labor.** MLflow does not train — it *remembers*. The compute
is the Ray genre-trainer on rogueone; MLflow is where its runs and models come to rest, and
this notebook is the window onto that. If your question is "how did training go / which model
do I load", you're in the right place. If it's "run the training", that's the trainer
container and its pinned Ray client, not here.

> **Read-only, throughout.** We searched experiments, runs, and the registry, and loaded a
> registered model — we created no run, no experiment, and no model version. MLflow's history
> is exactly as we found it.